# Species classification and sex determination

A logistic probe on frozen embeddings, then fusion of the photographs of one observation into one
prediction. Both tasks follow the same protocol:

* **Cross-validation over individuals.** Five folds split the *individuals*, stratified by class. The metrics use the pooled out-of-fold predictions.
* **Identity balancing.** Individuals give very unequal numbers of photographs. Every fit, the
  choice of the regularisation strength and every metric therefore give each individual the
  same weight, as in the re-identification analysis. 

## Settings

In [1]:
import tomllib
from pathlib import Path

ROOT = Path.cwd().parent
CONFIG = tomllib.loads((ROOT / "config.toml").read_text())

# Relative paths in config.toml start at the folder that holds it.
DATA_ROOT = ROOT / Path(CONFIG["data_root"]).expanduser()
DATABASE = DATA_ROOT / "seahorse_photo_database.xlsx"
SEGMENTED_COMPRESSED = DATA_ROOT / "segmented_compressed"
EMBEDDINGS = ROOT / Path(CONFIG["cache_dir"]).expanduser() / "embeddings"
for path in (DATABASE, SEGMENTED_COMPRESSED, EMBEDDINGS):
    assert path.exists(), f"Not found: {path}. Check the paths in {ROOT / 'config.toml'}."

## The image table

Every database photograph that has a segmented crop, with its labels. A photograph gives one
image: its best SAM 3 crop, the same crop the re-identification uses. A photograph has no crop
only when SAM 3 found no seahorse in it.

A photograph is named `SH<individual>_<observation>_<photo>`, so its name gives its individual
and its observation.

In [2]:
import json
import re

import pandas as pd
from itables import init_notebook_mode, show

init_notebook_mode(all_interactive=True)  # every DataFrame shown from here on is interactive

# 'SH024_002_007_det0_cropped.jpg' -> individual 'SH024', observation number '002'. The photo number
# and SAM 3's suffixes (_det<n>, _segmented, _mask, _cropped, _annotated) are optional.
PHOTO_NAME = re.compile(r"^(?P<identity>SH\d+)_(?P<number>\d+)(?:_\d+)?"
                        r"(?:_det\d+|_segmented|_mask|_cropped|_annotated)*(?:\.[^.]+)?$")
CROP = re.compile(r"^(?P<photograph>.+)_det(?P<detection>\d+)_cropped\.jpg$")

SPECIES_TO_LABEL = {"Hippocampus guttulatus": 0, "Hippocampus hippocampus": 1}
SPECIES_NAMES = {"H. guttulatus": "Hippocampus guttulatus",
                 "H. hippocampus": "Hippocampus hippocampus",
                 "Hippocampus guttulatus": "Hippocampus guttulatus",
                 "Hippocampus hippocampus": "Hippocampus hippocampus"}

photos = pd.read_excel(DATABASE, sheet_name="Photos")
photos["species"] = photos["species"].str.strip().map(SPECIES_NAMES)
photos = photos[photos["species"].notna()].reset_index(drop=True)

# The SAM 3 score of every crop. The paper's crops come from an earlier run of the same model and
# settings, which wrote summary.json; 1_segment_images.ipynb records its crops in detections.json.
# Both are read, so crops from the two runs can sit side by side; detections.json wins a tie.
crop_scores = {}
summary = DATA_ROOT / "summary.json"
if summary.exists():
    for record in json.loads(summary.read_text())["results"]:
        stem = Path(record["image_key"]).stem
        for index, score in enumerate(record.get("scores") or []):
            crop_scores[f"{stem}_det{index}_cropped.jpg"] = score
detections = DATA_ROOT / "detections.json"
if detections.exists():
    crop_scores.update({record["name"]: record["score"]
                        for record in json.loads(detections.read_text())})

# One crop per photograph: the SAM 3 detection with the highest score. SAM 3 lists its detections
# in the order of its internal queries, not by score, so _det0 is not always the best crop. Ties go
# to the lower detection number.
crop_files = pd.Series(sorted(path.name for path in SEGMENTED_COMPRESSED.glob("*.jpg")))
detected = crop_files.str.extract(CROP).assign(name=crop_files)
detected["detection"] = pd.to_numeric(detected["detection"])
detected["score"] = detected["name"].map(crop_scores)
several = detected.groupby("photograph")["name"].transform("size") > 1
assert detected.loc[several, "score"].notna().all(), "A photograph with several crops has no SAM 3 score"
best = (detected.dropna(subset=["photograph"])
        .sort_values(["photograph", "score", "detection"], ascending=[True, False, True])
        .drop_duplicates("photograph"))

# The crop of each photograph; a photograph without a crop is left out.
crop_by_photograph = dict(zip(best["photograph"], best["name"]))
photos["name"] = photos["proposed_new_filename"].map(lambda name: Path(name).stem).map(crop_by_photograph)
photos = photos.dropna(subset=["name"]).reset_index(drop=True)

# A name that does not follow the convention has no observation and is left out.
number = photos["proposed_new_filename"].str.extract(PHOTO_NAME)["number"]
photos["observation"] = photos["unique_id"] + "_" + number
photos = photos.dropna(subset=["observation"]).reset_index(drop=True)
photos["identity"] = photos["unique_id"]
photos["species_label"] = photos["species"].map(SPECIES_TO_LABEL).astype(int)
assert (photos.groupby("observation")["species"].nunique() == 1).all(), \
    "An observation carries more than one species label"
images = photos
show(images, scrollY="400px", scroller=True, maxBytes=0, showIndex=False)

## Embeddings

In [3]:
import numpy as np


def read_embeddings(path, names):
    """The vectors of the named images from an embedding cache, in order, scaled to length 1."""
    if not path.exists():
        raise FileNotFoundError(f"No embedding cache at {path}. Run 3_extract_embeddings.ipynb.")
    with np.load(path) as stored:
        row_by_name = {name: row for row, name in enumerate(stored["names"])}
        vectors = stored["embeddings"]
    missing = [name for name in dict.fromkeys(names) if name not in row_by_name]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images are missing from {path}, "
                                f"e.g. {missing[:3]}. Run 3_extract_embeddings.ipynb.")
    vectors = vectors[[row_by_name[name] for name in names]]
    corrupt = [names[row] for row in np.flatnonzero(~np.isfinite(vectors).all(axis=1))]
    if corrupt:
        raise ValueError(f"Non-finite embeddings in {path}, e.g. {corrupt[:3]}")
    return vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)


names = images["name"].tolist()
dinov3 = read_embeddings(EMBEDDINGS / "dinov3_vitl16_cls.npz", names)
bioclip = read_embeddings(EMBEDDINGS / "bioclip2.5_vith14.npz", names)
images["row"] = np.arange(len(images))  # the row of each image in dinov3 and bioclip
print(f"{len(images):,} images | {images['observation'].nunique():,} observations | "
      f"{images['identity'].nunique()} individuals")
print(f"DINOv3 {dinov3.shape} | BioCLIP 2.5 {bioclip.shape}")

30,500 images | 2,488 observations | 474 individuals
DINOv3 (30500, 1024) | BioCLIP 2.5 (30500, 1024)


## Species: *H. hippocampus* versus *H. guttulatus*

The positive class is *H. hippocampus*, 2% of the observations.  

**Task table.** `task_table` gives one row per image: its row in the embedding matrices, its
label, its individual, its observation and its weight. It checks that no observation mixes
labels.

**Weights.** `identity_balanced_weights` gives all rows of one individual one unit of weight in
total, so an animal photographed 252 times cannot outweigh a dozen animals seen once. Each class also gets the same total weight. 

**Folds.** `individual_folds` splits the individuals of a task table into folds, stratified
by class, and returns the images of each fold. The outer folds split the whole table. The inner
folds split the training images of one outer fold, to choose C.

In [ ]:
import gc

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
C_GRID = np.logspace(-4, 2, 49)  # inverse regularization strength 
OUTER_FOLDS = 5
INNER_FOLDS = 5


def identity_balanced_weights(identities, labels=None):
    """Row weights that give every individual (and, with labels, every class) the same weight.

    The weights sum to the number of rows.
    """
    _, inverse, counts = np.unique(identities, return_inverse=True, return_counts=True)
    weights = 1.0 / counts[inverse]
    if labels is not None:
        _, class_inverse = np.unique(labels, return_inverse=True)
        weights = weights / np.bincount(class_inverse, weights=weights)[class_inverse]
    return weights * (len(weights) / weights.sum())


def task_table(images, labels):
    """One row per image: its row in the embedding matrices, label, individual, observation and
    identity- and class-balanced weight."""
    table = images[["row", "identity", "observation"]].assign(label=np.asarray(labels, dtype=int))
    mixed = table.groupby("observation")["label"].nunique() > 1
    assert not mixed.any(), f"Observations that mix labels: {mixed.index[mixed].tolist()[:3]}"
    table["weight"] = identity_balanced_weights(table["identity"], table["label"])
    return table


def individual_folds(table, n_folds):
    """(train, test) parts of a task table, split over its individuals and stratified by class."""
    per_individual = table.groupby("identity")["label"]
    assert per_individual.nunique().eq(1).all(), "An individual carries two class labels"
    labels = per_individual.first()
    n_folds = min(n_folds, labels.value_counts().min())
    splitter = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    return [(table[table["identity"].isin(labels.index[train])],
             table[table["identity"].isin(labels.index[test])])
            for train, test in splitter.split(labels.index, labels)]


def fit_probe(features, part, c):
    """Standardise, then fit an L2-regularised logistic regression on part of a task table.

    The scaler takes the weights too. The L2 penalty acts in the standardised space, so an
    unweighted scale would let the most photographed animals set the regularisation.
    """
    probe = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, C=c, class_weight=None, random_state=SEED)),
    ])
    probe.fit(features[part["row"]], part["label"],
              scaler__sample_weight=part["weight"], classifier__sample_weight=part["weight"])
    # without this line, the fits of this notebook need over 20 GB of memory.
    gc.collect()
    return probe


def select_c(features, train):
    """The C of C_GRID with the best pooled inner-fold balanced accuracy on a training part."""
    folds = individual_folds(train, INNER_FOLDS)
    for core, _ in folds:
        assert core["label"].nunique() == 2, "An inner core part lacks a class"
    held_out = pd.concat([part for _, part in folds])

    scores = [balanced_accuracy_score(held_out["label"],
                                      np.concatenate([fit_probe(features, core, c).predict(features[part["row"]])
                                                      for core, part in folds]),
                                      sample_weight=held_out["weight"])
              for c in C_GRID]
    return float(C_GRID[np.argmax(scores)])  # argmax takes the first best: ties go to the smaller C


def out_of_fold_logits(features, table):
    """Each image's logit from a probe that never saw its animal, and the C of each fold."""
    logits = pd.Series(np.nan, index=table.index)
    selected_c = []
    for train, test in individual_folds(table, OUTER_FOLDS):
        c = select_c(features, train)
        logits.loc[test.index] = fit_probe(features, train, c).decision_function(features[test["row"]])
        selected_c.append(c)
    assert logits.notna().all(), "Some images received no out-of-fold logit"
    return logits.to_numpy(), selected_c


species = task_table(images, images["species_label"])
species_labels = species["label"].to_numpy()
observations, individuals = species["observation"].to_numpy(), species["identity"].to_numpy()

# About twelve minutes per backbone: five folds, each searching 49 values of C.
species_logits = {}
for backbone, features in [("DINOv3 ViT-L/16 CLS", dinov3), ("BioCLIP 2.5", bioclip)]:
    logits, selected_c = out_of_fold_logits(features, species)
    species_logits[backbone] = logits
    print(f"{backbone:22s} C per fold: {[round(c, 2) for c in selected_c]}")

### Fusing the photographs of an observation

The three rules read the same out-of-fold image logits; nothing is refitted.

* **Mean logit**: $P(Y=1) = \sigma(\text{mean logit})$. A near-duplicate photograph does not
  sharpen the posterior.
* **Logit sum**: $P(Y=1) = \sigma(\text{sum of logits})$, the rule under conditional
  independence. It grows more confident with each extra view.
* **Majority vote**: each image votes for its predicted class, and $P(Y=1)$ is the share of
  positive votes.

The two logit rules take the same decisions, so they have the same accuracy and differ only in
calibration.

### Metrics and intervals

Every metric weights each individual equally. Plain accuracy is left out: at these class
prevalences it reports the share of the majority class (0.973 for species), not performance.

**Calibration.** ECE is the weighted mean, over ten equal-width probability bins, of the gap
between the mean predicted probability and the rate of positives. MCE is the unweighted maximum
of the same gaps. A bin with one observation can therefore set MCE and barely move ECE. No library
takes sample weights here, so the code writes both out. Without weights it gives the `l1` and
`max` norms of `torchmetrics`.

**Intervals.** The percentile bootstrap resamples individuals with replacement and takes all
their rows, with their weights. An individual drawn twice counts twice.

### Table 3 — Backbone comparison, image and observation level

The observation rows fuse the *same* out-of-fold image logits by their mean; nothing is refitted.
The gain from image to observation is therefore the gain from combining views.

In [5]:
from functools import partial

from IPython.display import Markdown, display
from scipy.special import expit as sigmoid
from scipy.stats import bootstrap
from sklearn.metrics import brier_score_loss, f1_score, roc_auc_score

CALIBRATION_BINS = 10


def calibration_errors(y_true, prob, weight):
    """ECE and MCE over CALIBRATION_BINS equal-width bins, both identity-weighted."""
    edges = np.linspace(0.0, 1.0, CALIBRATION_BINS + 1)
    # Bins [0, 0.1), [0.1, 0.2), ..., [0.9, 1]: a probability of exactly 1.0 falls in the last bin.
    bin_index = np.digitize(prob, edges[1:-1])
    expected, maximum, total = 0.0, 0.0, weight.sum()
    for b in range(CALIBRATION_BINS):
        in_bin = bin_index == b
        bin_weight = weight[in_bin].sum()
        if bin_weight <= 0.0:
            continue
        gap = abs(np.average(prob[in_bin], weights=weight[in_bin])
                  - np.average(y_true[in_bin], weights=weight[in_bin]))
        expected += bin_weight / total * gap
        maximum = max(maximum, gap)
    return expected, maximum


CLASSIFICATION_METRICS = {
    "balanced_accuracy": lambda y, pred, prob, w: balanced_accuracy_score(y, pred, sample_weight=w),
    "macro_f1": lambda y, pred, prob, w: f1_score(y, pred, average="macro", zero_division=0,
                                                  sample_weight=w),
    "auroc": lambda y, pred, prob, w: (roc_auc_score(y, prob, sample_weight=w)
                                       if len(np.unique(y)) > 1 else float("nan")),
    "brier": lambda y, pred, prob, w: brier_score_loss(y, prob, sample_weight=w),
    "ece": lambda y, pred, prob, w: calibration_errors(y, prob, w)[0],
    "mce": lambda y, pred, prob, w: calibration_errors(y, prob, w)[1],
}


N_BOOT = 1000  # bootstrap draws for every interval
ALPHA = 0.05  # 95% intervals
PRECISION = 2  # decimals of every number in every table
DISCRIMINATION = ["Balanced acc. ↑", "Macro F1 ↑", "AUROC ↑"]
CALIBRATION = ["Brier ↓", "ECE ↓", "MCE ↓"]
SPECIES_DISCRIMINATION = ["Balanced acc. ↑", "Macro F1 ↑"]

METRIC_BY_HEADER = {"Balanced acc. ↑": "balanced_accuracy", "Macro F1 ↑": "macro_f1", "AUROC ↑": "auroc",
             "Brier ↓": "brier", "ECE ↓": "ece", "MCE ↓": "mce"}
ARROWS = str.maketrans({"↑": r"$\uparrow$", "↓": r"$\downarrow$"})


def shared_exponent(value):
    """The power of ten to factor out, or None when two decimals already show the value."""
    if value == 0 or abs(value) >= 0.5 * 10 ** -PRECISION:
        return None
    return int(np.floor(np.log10(abs(value))))


def format_ci(cell, scientific=False, latex=False):
    """A (point, low, high) cell as 'point [low, high]', or '-' without a point.

    With scientific=True, a point that two decimals would print as 0.00 shares one power of ten
    with its interval: '2.93 [0.00, 10.00]e-03', in LaTeX '2.93 [0.00, 10.00]${}\\times 10^{-3}$'.
    """
    point, low, high = cell
    if not np.isfinite(point):
        return r"\textemdash{}" if latex else "-"
    exponent = shared_exponent(point) if scientific else None
    scale = 1.0 if exponent is None else 10.0 ** exponent
    text = f"{point / scale:.{PRECISION}f}"
    if np.isfinite(low) and np.isfinite(high):
        text += f" [{low / scale:.{PRECISION}f}, {high / scale:.{PRECISION}f}]"
    if exponent is None:
        return text
    return text + (rf"${{}}\times 10^{{{exponent}}}$" if latex else f"e{exponent:+03d}")


def show_table(frame, caption, label, bold_best=True):
    """Display the table, sortable and searchable, then its booktabs LaTeX.

    The caption is LaTeX. The columns of (point, low, high) cells are right-aligned, and the
    calibration columns print in scientific notation. With bold_best, bold marks the best printed
    point of each column whose header ends in ↑ or ↓.
    """
    intervals = [column for column in frame if isinstance(frame[column].iloc[0], tuple)]
    formats = {column: partial(format_ci, scientific=column in CALIBRATION) for column in intervals}
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
    for column in intervals:
        if bold_best and column.endswith(("↑", "↓")):
            # Each point as format_ci prints it, so that two points equal in print tie.
            printed = frame[column].map(lambda cell: round(cell[0], PRECISION - (shared_exponent(cell[0]) or 0)
                                                           if column in CALIBRATION else PRECISION))
            best = printed.max() if column.endswith("↑") else printed.min()
            styles[column] = np.where(printed == best, "textbf:--rwrap;", "")
    latex = (frame.style.hide(axis="index")
             .format({column: partial(formatter, latex=True) for column, formatter in formats.items()})
             .apply(lambda _: styles, axis=None)
             .format_index(lambda header: header.translate(ARROWS), axis=1)
             .to_latex(hrules=True, caption=caption, label=label, position="htbp",
                       position_float="centering",
                       column_format="".join("r" if column in intervals else "l"
                                             for column in frame)))
    show(frame.assign(**{column: frame[column].map(formatter) for column, formatter in formats.items()}),
         showIndex=False, paging=False)
    display(Markdown(f"**`{label}`**\n\n```latex\n{latex}\n```"))


def predictions(observations, identities, labels, logits):
    """The predictions of the per-image probe and of the three fusion rules.

    A dict {method: frame}. Each frame has one row per image or per observation, with its
    individual, its label y_true, the decision pred and the probability prob.
    """
    images = pd.DataFrame({"observation": observations, "individual": identities,
                           "y_true": labels, "logit": logits})
    images["vote"] = (images["logit"] >= 0.0).astype(int)
    fused = images.groupby("observation", sort=False).agg(
        individual=("individual", "first"), y_true=("y_true", "first"),
        mean_logit=("logit", "mean"), sum_logit=("logit", "sum"), vote_share=("vote", "mean"),
    ).reset_index()
    fused_prob = {"Mean logit": sigmoid(fused["mean_logit"]), "Logit sum": sigmoid(fused["sum_logit"]),
                  "Majority vote": fused["vote_share"]}
    return {"Image (LR)": images.assign(pred=images["vote"], prob=sigmoid(images["logit"])),
            **{method: fused.assign(pred=(prob >= 0.5).astype(int), prob=prob)
               for method, prob in fused_prob.items()}}


def bootstrap_over_individuals(y_true, y_pred, y_prob, individuals, weight):
    """{metric: (point, low, high)}, with individuals resampled N_BOOT times.

    scipy resamples the individual numbers; the statistic takes all the rows of each drawn
    individual, with their weights. An individual drawn twice counts twice.
    """
    rows_by_individual = list(pd.Series(individuals).groupby(individuals).indices.values())

    def metrics(drawn):
        rows = np.concatenate([rows_by_individual[individual] for individual in drawn.astype(int)])
        return np.array([metric(y_true[rows], y_pred[rows], y_prob[rows], weight[rows])
                         for metric in CLASSIFICATION_METRICS.values()])

    everyone = np.arange(len(rows_by_individual))
    result = bootstrap((everyone,), metrics, vectorized=False, n_resamples=N_BOOT,
                       confidence_level=1 - ALPHA, method="percentile",
                       rng=np.random.default_rng(SEED))
    return dict(zip(CLASSIFICATION_METRICS, zip(metrics(everyone), *result.confidence_interval)))


def classification_table(entries, columns=DISCRIMINATION + CALIBRATION):
    """One row per method of entries, a dict {method: predictions frame}. Each cell is a
    (point, low, high) tuple.

    The calibration columns sit near 1e-3 on the species task, where two decimals would print
    0.00, so show_table prints them in scientific notation.
    """
    rows = []
    for method, frame in entries.items():
        individuals = frame["individual"].to_numpy()
        interval = bootstrap_over_individuals(frame["y_true"].to_numpy(), frame["pred"].to_numpy(),
                                              frame["prob"].to_numpy(), individuals,
                                              identity_balanced_weights(individuals))
        rows.append({"Method": method} | {column: interval[METRIC_BY_HEADER[column]] for column in columns})
    return pd.DataFrame(rows)


entries = {}
for backbone, logits in species_logits.items():
    methods = predictions(observations, individuals, species_labels, logits)
    entries[f"{backbone} (image)"] = methods["Image (LR)"]
    entries[f"{backbone} (observation)"] = methods["Mean logit"]

backbone_table = classification_table(entries, columns=SPECIES_DISCRIMINATION)
show_table(backbone_table, label="tab:species_backbone",
           caption='Backbone comparison at the image and observation levels (observation = mean-logit fusion) for species classification. All metrics are identity-balanced, with 95\\% individual-level bootstrap intervals over the 13 \\textit{H. hippocampus} and 461 \\textit{H. guttulatus} individuals.')

**`tab:species_backbone`**

```latex
\begin{table}[htbp]
\centering
\begin{tabular}{lrr}
\toprule
Method & Balanced acc. $\uparrow$ & Macro F1 $\uparrow$ \\
\midrule
DINOv3 ViT-L/16 CLS (image) & 0.91 [0.87, 0.94] & 0.79 [0.71, 0.84] \\
DINOv3 ViT-L/16 CLS (observation) & \textbf{0.97 [0.92, 1.00]} & \textbf{0.99 [0.95, 1.00]} \\
BioCLIP 2.5 (image) & 0.83 [0.76, 0.88] & 0.71 [0.64, 0.76] \\
BioCLIP 2.5 (observation) & 0.91 [0.82, 0.98] & 0.93 [0.87, 0.97] \\
\bottomrule
\end{tabular}
\caption{Backbone comparison at the image and observation levels (observation = mean-logit fusion) for species classification. All metrics are identity-balanced, with 95\% individual-level bootstrap intervals over the 13 \textit{H. hippocampus} and 461 \textit{H. guttulatus} individuals.}
\label{tab:species_backbone}
\end{table}

```

### Table 4 — Image fusion methods

In [6]:
species_table = classification_table(predictions(observations, individuals, species_labels,
                                                 species_logits["DINOv3 ViT-L/16 CLS"]))
show_table(species_table[["Method", *SPECIES_DISCRIMINATION, *CALIBRATION]],
           label="tab:species_fusion",
           caption='Comparison of image fusion methods with DINOv3 CLS, with the per-image logistic regression as a baseline, in species classification. All metrics are identity-balanced, with 95\\% individual-level bootstrap intervals. Calibration errors use ten equal-width probability bins. ECE: Expected Calibration Error. MCE: Maximum Calibration Error.')

# The text quotes AUROC once, so print it here.
print("AUROC, not tabulated:",
      {method: format_ci(cell) for method, cell in zip(species_table["Method"], species_table["AUROC ↑"])})

**`tab:species_fusion`**

```latex
\begin{table}[htbp]
\centering
\begin{tabular}{lrrrrr}
\toprule
Method & Balanced acc. $\uparrow$ & Macro F1 $\uparrow$ & Brier $\downarrow$ & ECE $\downarrow$ & MCE $\downarrow$ \\
\midrule
Image (LR) & 0.91 [0.87, 0.94] & 0.79 [0.71, 0.84] & 0.03 [0.02, 0.03] & 0.06 [0.06, 0.07] & 0.48 [0.38, 0.60] \\
Mean logit & 0.97 [0.92, 1.00] & \textbf{0.99 [0.95, 1.00]} & 0.01 [0.00, 0.01] & 0.04 [0.03, 0.04] & 0.51 [0.35, 0.51] \\
Logit sum & 0.97 [0.92, 1.00] & \textbf{0.99 [0.95, 1.00]} & \textbf{1.42 [0.03, 4.08]${}\times 10^{-3}$} & \textbf{1.34 [0.30, 4.33]${}\times 10^{-3}$} & 0.66 [0.27, 0.87] \\
Majority vote & \textbf{0.98 [0.93, 1.00]} & 0.98 [0.95, 1.00] & 0.01 [0.00, 0.01] & 0.03 [0.02, 0.03] & \textbf{0.34 [0.32, 0.50]} \\
\bottomrule
\end{tabular}
\caption{Comparison of image fusion methods with DINOv3 CLS, with the per-image logistic regression as a baseline, in species classification. All metrics are identity-balanced, with 95\% individual-level bootstrap intervals. Calibration errors use ten equal-width probability bins. ECE: Expected Calibration Error. MCE: Maximum Calibration Error.}
\label{tab:species_fusion}
\end{table}

```

AUROC, not tabulated: {'Image (LR)': '0.98 [0.97, 0.99]', 'Mean logit': '1.00 [1.00, 1.00]', 'Logit sum': '1.00 [1.00, 1.00]', 'Majority vote': '0.99 [0.98, 1.00]'}


## Sex: female versus male

The 63 individuals of unknown sex are left out. The folds, the nested choice of C and the identity
balancing do not change.

In [7]:
SEX_TO_LABEL = {"Female": 0, "Male": 1}

known_sex = images["sex"].isin(SEX_TO_LABEL)
sex = task_table(images[known_sex], images.loc[known_sex, "sex"].map(SEX_TO_LABEL))
sex_labels = sex["label"].to_numpy()
sex_observations = sex["observation"].to_numpy()
sex_individuals = sex["identity"].to_numpy()
sex_logits, selected_c = out_of_fold_logits(dinov3, sex)

print(f"{len(sex):,} images | {sex['observation'].nunique():,} observations | "
      f"{sex['identity'].nunique()} individuals")
print(f"C per fold: {[round(c, 2) for c in selected_c]}")

29,487 images | 2,415 observations | 411 individuals
C per fold: [0.0, 0.02, 0.0, 0.01, 0.01]


### Table 5 — Image fusion methods for sex

In [8]:
sex_table = classification_table(predictions(sex_observations, sex_individuals, sex_labels, sex_logits))
show_table(sex_table[["Method", *DISCRIMINATION, *CALIBRATION]],
           label="tab:sex_fusion",
           caption='Comparison of image fusion methods with DINOv3 CLS, with the per-image logistic regression as a baseline, in sex classification. All metrics are identity-balanced, with 95\\% individual-level bootstrap intervals. ECE: Expected Calibration Error. MCE: Maximum Calibration Error.')

**`tab:sex_fusion`**

```latex
\begin{table}[htbp]
\centering
\begin{tabular}{lrrrrrr}
\toprule
Method & Balanced acc. $\uparrow$ & Macro F1 $\uparrow$ & AUROC $\uparrow$ & Brier $\downarrow$ & ECE $\downarrow$ & MCE $\downarrow$ \\
\midrule
Image (LR) & 0.79 [0.77, 0.81] & 0.79 [0.77, 0.80] & 0.87 [0.85, 0.89] & 0.15 [0.14, 0.16] & \textbf{0.05 [0.03, 0.07]} & \textbf{0.10 [0.07, 0.16]} \\
Mean logit & \textbf{0.90 [0.87, 0.92]} & \textbf{0.89 [0.87, 0.92]} & \textbf{0.96 [0.94, 0.97]} & \textbf{0.08 [0.07, 0.10]} & 0.07 [0.05, 0.09] & 0.13 [0.12, 0.28] \\
Logit sum & \textbf{0.90 [0.87, 0.92]} & \textbf{0.89 [0.87, 0.92]} & 0.95 [0.93, 0.97] & 0.10 [0.08, 0.12] & 0.09 [0.07, 0.12] & 0.66 [0.51, 0.79] \\
Majority vote & 0.87 [0.84, 0.90] & 0.87 [0.84, 0.89] & 0.94 [0.93, 0.96] & 0.10 [0.08, 0.11] & \textbf{0.05 [0.04, 0.08]} & 0.17 [0.12, 0.26] \\
\bottomrule
\end{tabular}
\caption{Comparison of image fusion methods with DINOv3 CLS, with the per-image logistic regression as a baseline, in sex classification. All metrics are identity-balanced, with 95\% individual-level bootstrap intervals. ECE: Expected Calibration Error. MCE: Maximum Calibration Error.}
\label{tab:sex_fusion}
\end{table}

```